# EDA + Baselines — Telco Customer Churn

**Etapa 1 do Tech Challenge** — Entendimento e Preparação  
Cobre: exploração de dados, formulação do problema e modelos baseline registrados no MLflow.

In [ ]:
import sys

sys.path.insert(0, "..")

import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

from src.data.preprocessing import build_preprocessing_pipeline, load_data
from src.models.baseline import evaluate_model, get_baselines

SEED = 42
np.random.seed(SEED)
DATA_PATH = "../data/Telco_customer_churn.xlsx"

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100

## 1. Carregamento e Visão Geral

In [ ]:
X, y = load_data(DATA_PATH)
print(f"Shape: {X.shape}")
print("\nDistribuição do target:")
print(y.value_counts())
print(f"\nChurn rate: {y.mean():.1%}")

In [ ]:
X.info()

## 2. Valores Ausentes

In [ ]:
missing = X.isnull().sum()
missing = missing[missing > 0]
if missing.empty:
    print("Nenhum valor ausente nas features selecionadas.")
else:
    print(missing.sort_values(ascending=False))

## 3. Distribuição do Target

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
y.value_counts().plot(kind="bar", ax=ax, color=["steelblue", "tomato"])
ax.set_xticklabels(["Não Churn (0)", "Churn (1)"], rotation=0)
ax.set_title("Distribuição do Target")
ax.set_ylabel("Contagem")
plt.tight_layout()
plt.show()

## 4. Features Numéricas

In [ ]:
num_cols = ["Tenure Months", "Monthly Charges", "Total Charges"]
df_plot = X[num_cols].copy()
df_plot["Churn"] = y.values

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, num_cols):
    for churn_val, color, label in [(0, "steelblue", "Não Churn"), (1, "tomato", "Churn")]:
        subset = df_plot.loc[df_plot["Churn"] == churn_val, col].dropna()
        ax.hist(subset, bins=30, alpha=0.6, color=color, label=label, density=True)
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
X[num_cols].describe()

## 5. Features Categóricas

In [ ]:
cat_cols = ["Contract", "Internet Service", "Payment Method", "Gender"]

df_cat = X[cat_cols].copy()
df_cat["Churn"] = y.values

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, col in zip(axes.ravel(), cat_cols):
    churn_rate = df_cat.groupby(col)["Churn"].mean().sort_values(ascending=False)
    churn_rate.plot(kind="bar", ax=ax, color="tomato", alpha=0.8)
    ax.set_title(f"Taxa de Churn por {col}")
    ax.set_ylabel("Taxa de Churn")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
    ax.axhline(y.mean(), linestyle="--", color="gray", linewidth=1, label="Média global")
    ax.legend()
plt.tight_layout()
plt.show()

## 6. Correlação com o Target

In [ ]:
pipeline_for_corr = build_preprocessing_pipeline()
X_encoded = pipeline_for_corr.fit_transform(X, y)

from sklearn.compose import ColumnTransformer

ct = pipeline_for_corr.named_steps["features"]
ohe = ct.named_transformers_["cat"].named_steps["encoder"]
cat_feature_names = ohe.get_feature_names_out(["Gender", "Multiple Lines", "Internet Service",
    "Online Security", "Online Backup", "Device Protection",
    "Tech Support", "Streaming TV", "Streaming Movies", "Contract", "Payment Method"])

all_feature_names = (
    ["Tenure Months", "Monthly Charges", "Total Charges"]
    + ["Senior Citizen", "Partner", "Dependents", "Phone Service", "Paperless Billing"]
    + list(cat_feature_names)
)

corr_series = pd.Series(np.corrcoef(X_encoded.T, y.values)[:-1, -1], index=all_feature_names)
top_corr = corr_series.abs().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 5))
corr_series[top_corr.index].sort_values().plot(kind="barh", ax=ax, color="steelblue", alpha=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Top 15 features — correlação com Churn")
ax.set_xlabel("Correlação de Pearson")
plt.tight_layout()
plt.show()

## 7. Baselines com MLflow

In [ ]:
mlflow.set_experiment("telco-churn-eda")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

pipeline = build_preprocessing_pipeline()
X_train_t = pipeline.fit_transform(X_train, y_train)
X_test_t  = pipeline.transform(X_test)

results = {}
with mlflow.start_run(run_name="eda-baselines"):
    for name, model in get_baselines().items():
        with mlflow.start_run(run_name=name, nested=True):
            model.fit(X_train_t, y_train)
            y_pred  = model.predict(X_test_t)
            y_proba = model.predict_proba(X_test_t)[:, 1]
            metrics = evaluate_model(y_test.values, y_pred, y_proba)
            mlflow.log_params({"model": name})
            mlflow.log_metrics(metrics)
            results[name] = metrics
            print(f"{name:<26} AUC={metrics['roc_auc']:.4f}  F1={metrics['f1']:.4f}")

In [ ]:
results_df = pd.DataFrame(results).T[["roc_auc", "f1", "pr_auc", "accuracy"]]
results_df.sort_values("roc_auc", ascending=False).style.background_gradient(cmap="RdYlGn", axis=0)

## 8. Curva ROC — Comparação de Baselines

In [ ]:
from sklearn.metrics import auc, roc_curve

fig, ax = plt.subplots(figsize=(7, 5))
for name, model in get_baselines().items():
    model.fit(X_train_t, y_train)
    y_proba = model.predict_proba(X_test_t)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc(fpr, tpr):.3f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=0.8)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Curva ROC — Baselines")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()